# 03. X Feature 집계
- 02_preprocessing.ipynb 완료 후 실행
- 순서: 구종 그룹화 → X구간 feature 집계 → delta feature 적용 → 최종 feature table 저장

In [1]:
# ── 환경 감지 ──────────────────────────────────────────────
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
else:
    DRIVE = os.path.dirname(os.path.abspath('__file__'))

INTERIM_DIR  = os.path.join(DRIVE, '0_data', '2_interim')
FEATURE_DIR  = os.path.join(DRIVE, '0_data', '4_features')
DB_PATH      = os.path.join(DRIVE, '0_data', '2_interim', 'mlb_pitcher.duckdb')
os.makedirs(FEATURE_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'INTERIM_DIR: {INTERIM_DIR}')
print(f'FEATURE_DIR: {FEATURE_DIR}')
print(f'DB_PATH    : {DB_PATH}')

Mounted at /content/drive
환경: 코랩
INTERIM_DIR: /content/drive/MyDrive/투수 컨디션 예측 ML/0_data/2_interim
FEATURE_DIR: /content/drive/MyDrive/투수 컨디션 예측 ML/0_data/4_features
DB_PATH    : /content/drive/MyDrive/투수 컨디션 예측 ML/0_data/2_interim/mlb_pitcher.duckdb


In [2]:
# ── DuckDB 설치 ────────────────────────────────────────────
try:
    import duckdb
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'duckdb', '-q'])
    import duckdb

import pandas as pd
import numpy as np

print(f'DuckDB version: {duckdb.__version__}')

DuckDB version: 1.3.2


## 1. DuckDB 적재

In [3]:
# 연린 DuckDB 연결 및 parquet 등록 ────────────────
import duckdb
con = duckdb.connect(DB_PATH)

starters_path = os.path.join(INTERIM_DIR, "starters_all.parquet")
lookup_path   = os.path.join(INTERIM_DIR, "prev_season_lookup.parquet")

con.execute(f"""
    CREATE OR REPLACE TABLE starters AS
    SELECT * FROM read_parquet("{starters_path}")
""")

con.execute(f"""
    CREATE OR REPLACE TABLE prev_lookup AS
    SELECT * FROM read_parquet("{lookup_path}")
""")

n_rows = con.execute("SELECT COUNT(*) FROM starters").fetchone()[0]
print(f"starters 테이블: {n_rows:,}행 적재 완료")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

starters 테이블: 2,023,986행 적재 완료


In [4]:
# ── 구종 그룹 컬럼 추가 ────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TABLE starters AS
    SELECT *,
        CASE pitch_type
            WHEN 'FF' THEN 'Fastball'
            WHEN 'SI' THEN 'Fastball'
            WHEN 'FC' THEN 'Fastball'
            WHEN 'SL' THEN 'Breaking'
            WHEN 'CU' THEN 'Breaking'
            WHEN 'KC' THEN 'Breaking'
            WHEN 'CH' THEN 'Offspeed'
            WHEN 'FS' THEN 'Offspeed'
            ELSE 'Other'
        END AS pitch_group,
        CASE
            WHEN description IN ('called_strike','swinging_strike',
                                  'swinging_strike_blocked','foul',
                                  'foul_tip') THEN 1
            ELSE 0
        END AS is_strike
    FROM starters
""")
print('구종 그룹 + strike 컬럼 추가 완료')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

구종 그룹 + strike 컬럼 추가 완료


## 2. X 구간 설정

In [5]:
# 실험할 구간 조합 — 나중에 X 구간 실험(04)에서 전부 비교
# 여기서는 우선 batter/9 로 feature 생성
MODE = 'batter'   # 'pitch' | 'inning' | 'batter'
N    = 9

# X구간 필터 SQL 조건
if MODE == 'pitch':
    # 투구 순서 번호가 없으므로 pitch_number 기준 근사
    x_filter = f'pitch_number <= {N}'
elif MODE == 'inning':
    x_filter = f'inning <= {N}'
elif MODE == 'batter':
    # at_bat_number 기준: 경기별 최솟값 + N - 1
    x_filter = f'at_bat_number <= min_ab + {N} - 1'

print(f'X 구간 설정: mode={MODE}, n={N}')

X 구간 설정: mode=batter, n=9


## 3. X Feature 집계 (SQL)

In [6]:
# ── batter 모드: 경기별 at_bat_number 최솟값 먼저 계산 ────
con.execute("""
    CREATE OR REPLACE TABLE game_ab_min AS
    SELECT game_pk, pitcher, MIN(at_bat_number) AS min_ab
    FROM starters
    GROUP BY game_pk, pitcher
""")

# X구간 데이터 뷰 생성
con.execute("""
    CREATE OR REPLACE VIEW x_zone AS
    SELECT s.*
    FROM starters s
    JOIN game_ab_min g
      ON s.game_pk = g.game_pk AND s.pitcher = g.pitcher
    WHERE s.at_bat_number <= g.min_ab + 9 - 1
""")

n_x = con.execute('SELECT COUNT(*) FROM x_zone').fetchone()[0]
print(f'X구간 투구 수: {n_x:,}행')

X구간 투구 수: 486,759행


In [7]:
# ── 구종별 feature 집계 ────────────────────────────────────
# 구종 그룹별 (Fastball / Breaking / Offspeed) 평균 구속, 구속 분산, spin rate
pitch_group_features = con.execute("""
    SELECT
        game_pk,
        pitcher,
        season,
        pitch_group,
        COUNT(*)                          AS pitch_count,
        AVG(release_speed)                AS avg_speed,
        STDDEV(release_speed)             AS std_speed,
        AVG(release_spin_rate)            AS avg_spin,
        AVG(release_extension)            AS avg_ext,
        AVG(release_pos_x)                AS avg_pos_x,
        AVG(release_pos_z)                AS avg_pos_z
    FROM x_zone
    WHERE pitch_group != 'Other'
    GROUP BY game_pk, pitcher, season, pitch_group
""").df()

print(f'구종별 집계: {len(pitch_group_features):,}행')
pitch_group_features.head()

구종별 집계: 60,027행


,game_pk,pitcher,season,pitch_group,pitch_count,avg_speed,std_speed,avg_spin,avg_ext,avg_pos_x,avg_pos_z
0,633017,592314,2021,Fastball,16,92.406250,1.324371,2073.250000,6.118750,-2.012500,5.921875
1,633017,605135,2021,Fastball,13,89.653846,1.981420,2138.846154,6.453846,-1.222308,5.577692
2,633008,677960,2021,Breaking,2,86.900000,0.424264,2212.500000,5.600000,1.760000,5.675000
3,633006,642232,2021,Fastball,9,82.966667,1.176860,2085.333333,6.466667,3.142222,5.053333
4,633006,607644,2021,Fastball,4,92.650000,0.191485,2395.500000,6.125000,0.562500,6.707500


In [8]:
# ── 경기 단위 전체 feature 집계 ────────────────────────────
game_features = con.execute("""
    SELECT
        game_pk,
        pitcher,
        season,
        -- 전체 투구 수
        COUNT(*)                                        AS total_pitches,
        -- 전체 평균 구속 / 분산
        AVG(release_speed)                              AS avg_speed_all,
        STDDEV(release_speed)                           AS std_speed_all,
        -- Strike ratio
        AVG(is_strike)                                  AS strike_ratio,
        -- 구종 비율
        AVG(CASE WHEN pitch_group = 'Fastball' THEN 1.0 ELSE 0.0 END) AS fastball_ratio,
        AVG(CASE WHEN pitch_group = 'Breaking' THEN 1.0 ELSE 0.0 END) AS breaking_ratio,
        AVG(CASE WHEN pitch_group = 'Offspeed' THEN 1.0 ELSE 0.0 END) AS offspeed_ratio,
        -- 릴리스 포인트
        AVG(release_pos_x)                              AS avg_pos_x,
        AVG(release_pos_z)                              AS avg_pos_z,
        AVG(release_extension)                          AS avg_ext,
        -- arm angle
        AVG(arm_angle)                                  AS avg_arm_angle
    FROM x_zone
    GROUP BY game_pk, pitcher, season
""").df()

print(f'경기 단위 feature: {len(game_features):,}행')
game_features.head()

경기 단위 feature: 23,230행


,game_pk,pitcher,season,total_pitches,avg_speed_all,std_speed_all,strike_ratio,fastball_ratio,breaking_ratio,offspeed_ratio,avg_pos_x,avg_pos_z,avg_ext,avg_arm_angle
0,633793,664285,2021,9,85.544444,7.755016,0.444444,0.555556,0.444444,0.000000,1.077778,5.830000,5.733333,44.255556
1,633788,592662,2021,16,92.787500,2.600224,0.625000,0.500000,0.500000,0.000000,2.532500,6.048750,6.306250,46.856250
2,633788,607074,2021,19,89.742105,5.194261,0.526316,0.421053,0.263158,0.315789,2.266316,6.521579,6.026316,45.547368
3,633778,669456,2021,23,87.856522,4.578887,0.478261,0.434783,0.565217,0.000000,-1.195652,5.613043,6.669565,44.713043
4,633777,605200,2021,25,86.520000,3.125567,0.400000,0.840000,0.000000,0.160000,-1.714800,5.376000,6.092000,36.654167


In [9]:
# ── 구종별 feature → wide format으로 pivot ────────────────
pivot = pitch_group_features.pivot_table(
    index=['game_pk', 'pitcher', 'season'],
    columns='pitch_group',
    values=['avg_speed', 'std_speed', 'avg_spin', 'avg_ext', 'avg_pos_x', 'avg_pos_z'],
    aggfunc='first'
)
# 컬럼명 평탄화: avg_speed_Fastball 형태
pivot.columns = [f'{v}_{g}' for v, g in pivot.columns]
pivot = pivot.reset_index()

# 경기 단위 feature와 병합
features = game_features.merge(pivot, on=['game_pk', 'pitcher', 'season'], how='left')
print(f'병합 후: {len(features):,}행  |  컬럼 수: {len(features.columns)}')
print(features.columns.tolist())

병합 후: 23,230행  |  컬럼 수: 32
['game_pk', 'pitcher', 'season', 'total_pitches', 'avg_speed_all', 'std_speed_all', 'strike_ratio', 'fastball_ratio', 'breaking_ratio', 'offspeed_ratio', 'avg_pos_x', 'avg_pos_z', 'avg_ext', 'avg_arm_angle', 'avg_ext_Breaking', 'avg_ext_Fastball', 'avg_ext_Offspeed', 'avg_pos_x_Breaking', 'avg_pos_x_Fastball', 'avg_pos_x_Offspeed', 'avg_pos_z_Breaking', 'avg_pos_z_Fastball', 'avg_pos_z_Offspeed', 'avg_speed_Breaking', 'avg_speed_Fastball', 'avg_speed_Offspeed', 'avg_spin_Breaking', 'avg_spin_Fastball', 'avg_spin_Offspeed', 'std_speed_Breaking', 'std_speed_Fastball', 'std_speed_Offspeed']


## 4. Delta Feature 적용 (직전 시즌 대비 편차)

In [10]:
# lookup table 로드
lookup = pd.read_parquet(os.path.join(INTERIM_DIR, 'prev_season_lookup.parquet'))

FASTBALL = ['FF', 'SI', 'FC']
BREAKING = ['SL', 'CU', 'KC']
OFFSPEED = ['CH', 'FS']

# speed / spin / ext / pos_x / pos_z 기준값을 구종 그룹별로 집계
lookup_pivot = lookup[['pitcher', 'season']].drop_duplicates().set_index(['pitcher', 'season'])

for metric, prefix in [
    ('prev_avg_speed', 'prev_speed'),
    ('prev_avg_spin',  'prev_spin'),
    ('prev_avg_ext',   'prev_ext'),
    ('prev_avg_pos_x', 'prev_pos_x'),
    ('prev_avg_pos_z', 'prev_pos_z'),
]:
    if metric not in lookup.columns:
        print(f'[skip] {metric} 컬럼 없음 — lookup table 재생성 필요')
        continue
    pivot = lookup.pivot_table(
        index=['pitcher', 'season'],
        columns='pitch_type',
        values=metric,
        aggfunc='mean'
    )
    existing = pivot.columns.tolist()
    for group, types in [('Fastball', FASTBALL), ('Breaking', BREAKING), ('Offspeed', OFFSPEED)]:
        col = f'{prefix}_{group}'
        cols = [c for c in types if c in existing]
        lookup_pivot[col] = pivot[cols].mean(axis=1) if cols else pd.NA

lookup_pivot = lookup_pivot.reset_index()

# features 와 병합
features = features.merge(lookup_pivot, on=['pitcher', 'season'], how='left')

print('lookup 병합 완료')
print(f'추가된 기준값 컬럼: {[c for c in lookup_pivot.columns if c not in ["pitcher", "season"]]}')
lookup_pivot.head()

lookup 병합 완료
추가된 기준값 컬럼: ['prev_speed_Fastball', 'prev_speed_Breaking', 'prev_speed_Offspeed', 'prev_spin_Fastball', 'prev_spin_Breaking', 'prev_spin_Offspeed', 'prev_ext_Fastball', 'prev_ext_Breaking', 'prev_ext_Offspeed', 'prev_pos_x_Fastball', 'prev_pos_x_Breaking', 'prev_pos_x_Offspeed', 'prev_pos_z_Fastball', 'prev_pos_z_Breaking', 'prev_pos_z_Offspeed']


,pitcher,season,prev_speed_Fastball,prev_speed_Breaking,prev_speed_Offspeed,prev_spin_Fastball,prev_spin_Breaking,prev_spin_Offspeed,prev_ext_Fastball,prev_ext_Breaking,prev_ext_Offspeed,prev_pos_x_Fastball,prev_pos_x_Breaking,prev_pos_x_Offspeed,prev_pos_z_Fastball,prev_pos_z_Breaking,prev_pos_z_Offspeed
0,425794,2022,87.637845,73.459556,82.729744,2290.101237,2838.456754,1714.097938,6.544272,6.358301,6.495897,-1.186796,-1.175512,-1.333487,6.264548,6.368832,6.222359
1,425794,2023,86.962919,72.929479,82.18254,2282.202272,2775.230448,1763.634921,6.48552,6.288646,6.424339,-1.130936,-1.064969,-1.314709,6.277558,6.387771,6.223757
2,425794,2024,85.119669,71.521195,81.416471,2213.365026,2643.34,1663.211765,6.519513,6.3266,6.511765,-1.247079,-1.239846,-1.342471,6.222179,6.284046,6.139294
3,425844,2022,87.433462,77.371581,86.343705,2261.858798,2355.070078,1590.293478,5.961865,5.868436,5.909892,-1.357847,-1.223463,-1.374712,6.351771,6.355472,6.310576
4,425844,2023,88.142421,76.869832,86.468984,2284.555411,2440.192641,1643.493298,5.945932,5.847376,5.916845,-1.646031,-1.530116,-1.825642,6.241301,6.145417,6.142861


In [11]:
# delta feature 계산: 오늘값 - 직전 시즌 기준값 (구종 그룹별)
for metric, today_prefix, prev_prefix in [
    ('speed', 'avg_speed', 'prev_speed'),
    ('spin',  'avg_spin',  'prev_spin'),
    ('ext',   'avg_ext',   'prev_ext'),
    ('pos_x', 'avg_pos_x', 'prev_pos_x'),
    ('pos_z', 'avg_pos_z', 'prev_pos_z'),
]:
    for group in ['Fastball', 'Breaking', 'Offspeed']:
        today_col = f'{today_prefix}_{group}'
        prev_col  = f'{prev_prefix}_{group}'
        delta_col = f'delta_{metric}_{group}'

        if today_col in features.columns and prev_col in features.columns:
            features[delta_col] = features[today_col] - features[prev_col]
            valid = features[delta_col].notna().sum()
            print(f'{delta_col}: {valid:,}개 유효값')
        else:
            print(f'{delta_col}: 컬럼 없음 ({today_col} or {prev_col})')

# 2021 시즌은 직전 시즌(2020) 데이터 없으므로 delta = NaN → 정상
print(f'\n2021 시즌 delta_speed_Fastball NaN 비율: '
      f'{features[features["season"]==2021]["delta_speed_Fastball"].isna().mean():.1%} (정상)')

delta_speed_Fastball: 15,046개 유효값
delta_speed_Breaking: 12,383개 유효값
delta_speed_Offspeed: 10,515개 유효값
delta_spin_Fastball: 15,015개 유효값
delta_spin_Breaking: 12,354개 유효값
delta_spin_Offspeed: 10,482개 유효값
delta_ext_Fastball: 15,034개 유효값
delta_ext_Breaking: 12,371개 유효값
delta_ext_Offspeed: 10,506개 유효값
delta_pos_x_Fastball: 15,046개 유효값
delta_pos_x_Breaking: 12,383개 유효값
delta_pos_x_Offspeed: 10,515개 유효값
delta_pos_z_Fastball: 15,046개 유효값
delta_pos_z_Breaking: 12,383개 유효값
delta_pos_z_Offspeed: 10,515개 유효값

2021 시즌 delta_speed_Fastball NaN 비율: 100.0% (정상)


## 5. Y값 병합 및 최종 저장

In [12]:
# 02에서 계산한 Y값 로드
y_path = os.path.join(INTERIM_DIR, 'y_woba_batter9.parquet')
y_df   = pd.read_parquet(y_path)[['game_pk', 'pitcher', 'season', 'y_woba']]

# X feature + Y 병합
final = features.merge(y_df, on=['game_pk', 'pitcher', 'season'], how='inner')

# Y 결측 제거
final = final.dropna(subset=['y_woba'])

print(f'최종 데이터셋: {len(final):,}행  |  컬럼 수: {len(final.columns)}')
print(f'\ny_woba 분포:')
print(final['y_woba'].describe())

최종 데이터셋: 23,141행  |  컬럼 수: 63

y_woba 분포:
count    23141.000000
mean         0.327652
std          0.137912
min          0.000000
25%          0.230600
50%          0.316700
75%          0.410000
max          1.057100
Name: y_woba, dtype: float64


In [13]:
# ── feature 목록 확인 ──────────────────────────────────────
meta_cols    = ['game_pk', 'pitcher', 'season', 'y_woba']
feature_cols = [c for c in final.columns if c not in meta_cols]

print(f'feature 수: {len(feature_cols)}')
for c in feature_cols:
    na_rate = final[c].isna().mean()
    print(f'  {c:<35} NaN: {na_rate:.1%}')

feature 수: 59
  total_pitches                       NaN: 0.0%
  avg_speed_all                       NaN: 0.0%
  std_speed_all                       NaN: 0.0%
  strike_ratio                        NaN: 0.0%
  fastball_ratio                      NaN: 0.0%
  breaking_ratio                      NaN: 0.0%
  offspeed_ratio                      NaN: 0.0%
  avg_pos_x                           NaN: 0.0%
  avg_pos_z                           NaN: 0.0%
  avg_ext                             NaN: 0.1%
  avg_arm_angle                       NaN: 0.7%
  avg_ext_Breaking                    NaN: 15.6%
  avg_ext_Fastball                    NaN: 0.1%
  avg_ext_Offspeed                    NaN: 26.1%
  avg_pos_x_Breaking                  NaN: 15.5%
  avg_pos_x_Fastball                  NaN: 0.0%
  avg_pos_x_Offspeed                  NaN: 26.1%
  avg_pos_z_Breaking                  NaN: 15.5%
  avg_pos_z_Fastball                  NaN: 0.0%
  avg_pos_z_Offspeed                  NaN: 26.1%
  avg_speed_Breaking

In [14]:
# ── 저장 ──────────────────────────────────────────────────
out = os.path.join(FEATURE_DIR, 'features_batter9.parquet')
final.to_parquet(out, index=False)
print(f'저장 완료 → {out}')

# DuckDB에도 등록
con.execute(f"""
    CREATE OR REPLACE TABLE features AS
    SELECT * FROM read_parquet('{out}')
""")
print('DuckDB features 테이블 등록 완료')

final.head()

저장 완료 → /content/drive/MyDrive/투수 컨디션 예측 ML/0_data/4_features/features_batter9.parquet
DuckDB features 테이블 등록 완료


,game_pk,pitcher,season,total_pitches,avg_speed_all,std_speed_all,strike_ratio,fastball_ratio,breaking_ratio,offspeed_ratio,...,delta_ext_Fastball,delta_ext_Breaking,delta_ext_Offspeed,delta_pos_x_Fastball,delta_pos_x_Breaking,delta_pos_x_Offspeed,delta_pos_z_Fastball,delta_pos_z_Breaking,delta_pos_z_Offspeed,y_woba
0,633793,664285,2021,9,85.544444,7.755016,0.444444,0.555556,0.444444,0.000000,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.2269
1,633788,592662,2021,16,92.787500,2.600224,0.625000,0.500000,0.500000,0.000000,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.2667
2,633788,607074,2021,19,89.742105,5.194261,0.526316,0.421053,0.263158,0.315789,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.3559
3,633778,669456,2021,23,87.856522,4.578887,0.478261,0.434783,0.565217,0.000000,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.2714
4,633777,605200,2021,25,86.520000,3.125567,0.400000,0.840000,0.000000,0.160000,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0600


In [15]:
# ── 시즌별 샘플 수 확인 ────────────────────────────────────
con.execute("""
    SELECT season, COUNT(*) AS n_games
    FROM features
    GROUP BY season
    ORDER BY season
""").df()

,season,n_games
0,2021,4575
1,2022,4647
2,2023,4594
3,2024,4651
4,2025,4674


In [16]:
con.close()
print('DuckDB 연결 종료')

DuckDB 연결 종료
